# FHIR Encounter Data Quality Profiling

## Purpose

In this notebook, I profile the Silver FHIR Encounter dataset and define the
data-quality rules that will later be enforced directly inside the Lakeflow
Bronze-to-Silver transformation.

I am not creating another cleaned Encounter table in this notebook.

The purpose of this stage is to:

- identify incomplete or logically invalid Encounter records
- validate required relational identifiers
- inspect optional Practitioner and Organization references
- validate Encounter timing consistency
- inspect Encounter status and class domains
- classify rules as warning, drop, or fail
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.fhir_encounter`

### Production design

The final production flow will be:

Bronze  
↓  
FHIR Encounter transformation  
+  
Lakeflow expectations  
↓  
Validated Silver Encounter  
↓  
Gold

### Data-quality approach

FHIR Encounter records contain both required and optional relationships.

I treat the Encounter ID and Patient ID as critical relational fields, while
Practitioner, Organization, and Encounter type completeness are monitored
without automatically rejecting the record.

I also validate that Encounter timestamps remain logically consistent.

In [0]:
# loading the current Silver Encounter dataset for quality profiling.

from pyspark.sql import functions as F

ENCOUNTER_TABLE = "health_insurance.silver.fhir_encounter"

encounter_df = spark.table(ENCOUNTER_TABLE)

print(f"Encounter rows: {encounter_df.count():,}")

encounter_df.printSchema()

display(encounter_df.limit(10))

In [0]:
# defining candidate Encounter quality rules by severity.

ENCOUNTER_WARN_RULES = {
    "practitioner_reference_present":
        "practitioner_id IS NOT NULL",

    "organization_reference_present":
        "organization_id IS NOT NULL",

    "encounter_type_present":
        "encounter_type IS NOT NULL",

    "status_present":
        "status IS NOT NULL",

    "encounter_class_present":
        "encounter_class IS NOT NULL",

    "start_datetime_present":
        "start_datetime IS NOT NULL"
}


ENCOUNTER_DROP_RULES = {
    "encounter_id_present":
        "encounter_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL"
}


ENCOUNTER_FAIL_RULES = {
    "encounter_timeline_valid":
        """
        start_datetime IS NULL
        OR end_datetime IS NULL
        OR end_datetime >= start_datetime
        """
}

In [0]:
# measuring how many Encounter rows violate each candidate quality rule.

def profile_rules(df, rules, severity):

    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(
                f"NOT ({condition}) OR ({condition}) IS NULL"
            )
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition.strip(),
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
# profiling all proposed Encounter quality rules.

encounter_quality_results = []

encounter_quality_results += profile_rules(
    encounter_df,
    ENCOUNTER_WARN_RULES,
    "WARN"
)

encounter_quality_results += profile_rules(
    encounter_df,
    ENCOUNTER_DROP_RULES,
    "DROP"
)

encounter_quality_results += profile_rules(
    encounter_df,
    ENCOUNTER_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the Encounter quality profile as a structured result.

encounter_quality_profile_df = spark.createDataFrame(
    encounter_quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    encounter_quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# inspecting the actual Encounter status and class values
# before finalizing controlled-domain quality rules.

display(
    encounter_df
    .groupBy(
        "status",
        "encounter_class"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

In [0]:
# inspecting Encounter timing and relationship completeness.

encounter_df.select(
    F.count("*").alias("total_encounters"),

    F.sum(
        F.col("start_datetime").isNull().cast("int")
    ).alias("missing_start"),

    F.sum(
        F.col("end_datetime").isNull().cast("int")
    ).alias("missing_end"),

    F.sum(
        (
            F.col("end_datetime")
            < F.col("start_datetime")
        ).cast("int")
    ).alias("end_before_start"),

    F.sum(
        F.col("practitioner_id").isNull().cast("int")
    ).alias("missing_practitioner"),

    F.sum(
        F.col("organization_id").isNull().cast("int")
    ).alias("missing_organization"),

    F.sum(
        F.col("encounter_type").isNull().cast("int")
    ).alias("missing_encounter_type")
).show()

In [0]:
# defining the finalized Encounter quality contract.

ENCOUNTER_WARN_RULES = {
    "practitioner_reference_present":
        "practitioner_id IS NOT NULL",

    "organization_reference_present":
        "organization_id IS NOT NULL",

    "encounter_type_present":
        "encounter_type IS NOT NULL",

    "recognized_status":
        "status IN ('FINISHED')",

    "recognized_encounter_class":
        "encounter_class IN ('AMB', 'EMER', 'IMP')",

    "start_datetime_present":
        "start_datetime IS NOT NULL"
}


ENCOUNTER_DROP_RULES = {
    "encounter_id_present":
        "encounter_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL"
}


ENCOUNTER_FAIL_RULES = {
    "encounter_timeline_valid":
        """
        start_datetime IS NULL
        OR end_datetime IS NULL
        OR end_datetime >= start_datetime
        """
}

## Publish approved Encounter quality rules

I have completed the Encounter quality profiling and finalized the approved
WARN, DROP, and FAIL rules.

I now publish these rules directly into the central Unity Catalog governance
table so the Lakeflow pipeline can retrieve them dynamically.

This keeps the published Encounter quality contract connected to the notebook
where I developed and validated it.

In [0]:
# converting the finalized Encounter quality contract
# into rows for the central governance repository.

from pyspark.sql import functions as F

def build_rule_rows(dataset, severity, rules, description, source_notebook):

    return [
        (
            dataset,
            rule_name,
            constraint.strip(),
            severity,
            True,
            description,
            source_notebook,
            "data_engineering",
            1
        )
        for rule_name, constraint in rules.items()
    ]

In [0]:
# preparing all approved Encounter rules for publication.

encounter_rule_rows = []

encounter_rule_rows += build_rule_rows(
    "encounter",
    "WARN",
    ENCOUNTER_WARN_RULES,
    "FHIR Encounter quality monitoring rule",
    "04-data-quality/04_encounter_quality_profile"
)

encounter_rule_rows += build_rule_rows(
    "encounter",
    "DROP",
    ENCOUNTER_DROP_RULES,
    "FHIR Encounter record validity rule",
    "04-data-quality/04_encounter_quality_profile"
)

encounter_rule_rows += build_rule_rows(
    "encounter",
    "FAIL",
    ENCOUNTER_FAIL_RULES,
    "Critical FHIR Encounter pipeline rule",
    "04-data-quality/04_encounter_quality_profile"
)

In [0]:
# creating the Encounter quality-rule publication DataFrame.

encounter_rules_df = (
    spark.createDataFrame(
        encounter_rule_rows,
        [
            "dataset",
            "rule_name",
            "constraint",
            "severity",
            "is_active",
            "description",
            "source_notebook",
            "owner",
            "version"
        ]
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(encounter_rules_df)

In [0]:
#  exposing the finalized Encounter rules
# as a temporary view for idempotent publishing.

encounter_rules_df.createOrReplaceTempView(
    "encounter_quality_rule_updates"
)

In [0]:
%sql
-- publishing the approved Encounter rules
-- directly from this profiling notebook.

MERGE INTO health_insurance.governance.quality_rules AS target

USING encounter_quality_rule_updates AS source

ON target.dataset = source.dataset
AND target.rule_name = source.rule_name

WHEN MATCHED THEN UPDATE SET

    target.constraint = source.constraint,
    target.severity = source.severity,
    target.is_active = source.is_active,
    target.description = source.description,
    target.source_notebook = source.source_notebook,
    target.owner = source.owner,

    target.version =
        CASE
            WHEN target.constraint <> source.constraint
              OR target.severity <> source.severity
            THEN COALESCE(target.version, 1) + 1
            ELSE target.version
        END,

    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN INSERT (
    dataset,
    rule_name,
    constraint,
    severity,
    is_active,
    description,
    source_notebook,
    owner,
    version,
    created_at,
    updated_at
)

VALUES (
    source.dataset,
    source.rule_name,
    source.constraint,
    source.severity,
    source.is_active,
    source.description,
    source.source_notebook,
    source.owner,
    source.version,
    source.created_at,
    source.updated_at
);

In [0]:
%sql
-- verifying the Encounter rules published by this notebook.

SELECT
    dataset,
    rule_name,
    severity,
    constraint,
    source_notebook,
    version,
    is_active,
    updated_at
FROM health_insurance.governance.quality_rules
--WHERE dataset = 'encounter'
ORDER BY severity, rule_name;